# Team Selous — RAG + Fine-Tuned SLM for Clinical Q&A

**Goal:** answer short clinical quick-reference questions so that the generated
`Answer` is as close as possible (low **mean character-level Levenshtein
distance**) to the hidden reference answer.

### Why the previous notebook scored poorly

The original `team_selous_finetune` notebook fine-tuned the model on
`question -> reference_answer` pairs **with no retrieved context at all**
("Option A: fine-tune first, RAG comes later as a separate layer" — RAG was
never actually added back in). Two things follow from that:

1. With ~40-80 training examples the model has almost nothing to generalise
   from, so on unseen test questions it has to guess the answer purely from
   its own fine-tuned weights.
2. Even a good guess that is phrased differently from the reference (extra
   words, different order, a longer sentence) gets punished hard by
   character-level Levenshtein distance.

### What this notebook does differently

It follows the pipeline you sketched:

```
User question
      ↓
Question understanding
      ↓
Identify/derive: Topic · Population · Care setting
      ↓
Query embedding  (E5, "query: " prefix)
      ↓
Metadata-aware retrieval
      ↓
Combined ChromaDB knowledge base  (kb_combined)
      ↓
Top relevant evidence/chunks
      ↓
Safety-aware SLM  (fine-tuned WITH retrieved evidence in its prompt)
      ↓
Post-generation safety checks
      ↓
Final answer + triage-review flag
```

Concretely:

* We build (or load) a **ChromaDB** knowledge base, embedded with
  `intfloat/e5-small-v2` using the asymmetric `passage:` / `query:` prefixes,
  exactly as specified in Rita's integration notes.
* Retrieval is **metadata-aware**: `Topic` is used as a strict filter,
  `Population` / `Care_Setting` are used to *rerank* results (they can be
  multi-valued, so we don't force an exact-match filter on them).
* The SLM (Qwen3, via Unsloth) is **fine-tuned to read retrieved evidence and
  answer in the same terse style as the reference answers** — so training and
  inference use the exact same prompt shape. This is the main fix versus the
  original notebook.
* Generated text goes through a small, transparent **post-processing /
  clean-up step** (strip preamble, cut to one clipped clause) before scoring,
  because Levenshtein distance is very sensitive to rambling output.
* A lightweight **safety-review flag** is computed for high-risk topics
  (emergency triage, medication safety) and reported separately — it is
  **not** appended to the scored `Answer` column, so it can't hurt the metric,
  but it gives a human reviewer a heads-up on cases worth double-checking.

Every section below is a short, self-contained cell with a comment explaining
*what* it does and *why*, so you (or a teammate) can follow the whole flow
without needing to have written it.


## 0. Install dependencies

Run this once. On Kaggle/Colab, uncomment the `!pip install` lines. If you're
running locally with everything already installed, just skip / leave
commented.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#Commented out so this cell is safe to "Run All" more than once.
#Uncomment on a fresh Kaggle / Colab runtime.

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

#Extra packages this notebook needs on top of the original one:
!pip install chromadb sentence-transformers


## 1. Config

Every path / hyperparameter the notebook uses lives here, in one place, so
you can tune things without hunting through the whole file.


In [ ]:
import os

CONFIG = {
    # --- data ---
    "train_csv": "/content/drive/MyDrive/TRI AI/train_qa",
    "test_csv": "/content/drive/MyDrive/TRI AI/test_questions.csv",     # no reference_answer column
    "val_fraction": 0.15,
    "random_state": 3407,

    # --- embeddings / retrieval ---
    "embed_model_name": "intfloat/e5-small-v2",   # per Rita's integration notes
    "chroma_path": "/content/drive/MyDrive/TRI AI/kb_comparison_chroma",
    "collection_name": "kb_new",
    "n_style_examples": 2,          # # of nearest short Q&A exemplars shown as style guide
    "near_dup_threshold": 0.92,     # cosine sim above this -> just copy the neighbour's answer

    # --- base model (Unsloth) ---
    "model_name": "unsloth/Qwen3-14B",
    # If training/inference is too slow/OOM on a free T4, drop to:
    # "model_name": "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    "max_seq_length": 2048,
    "load_in_4bit": True,

    # --- LoRA ---
    # Raised from 16 -> 32: a 14B model has a strong default "helpful assistant"
    # voice; more LoRA capacity gives fine-tuning more room to override it.
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.0,

    # --- training ---
    # With train_on_responses_only (added below) loss is computed on the tiny
    # answer span only, so it converges much faster per step -> more epochs
    # than before is both affordable and necessary on ~70 examples.
    "num_train_epochs": 14,
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "general_regularizer_fraction": 0.0,  # was 0.20 — was diluting the style signal

    # --- generation ---
    "max_new_tokens": 32,

    # --- misc ---
    "save_model": False,   # flip to True to save LoRA adapters at the end
}

print("Config loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 2. Load the training data

`train_qa.csv` has one row per question, with the fields the diagram calls
"Topic / Population / Care setting" already provided as columns
(`topic`, `population`, `care_setting`), plus the `reference_answer` we're
trying to match and a `document_id` linking related questions back to the
same source document.


In [4]:
import pandas as pd

qa = pd.read_csv(CONFIG["train_csv"])
print("Shape:", qa.shape)
print("Columns:", list(qa.columns))
print("Topics:", sorted(qa["topic"].unique()))
qa.head()


Shape: (83, 7)
Columns: ['question', 'topic', 'care_setting', 'population', 'document_id', 'reference_answer', 'QuestionId']
Topics: ['chronic_disease', 'emergency_triage', 'infectious_disease', 'maternal_health', 'medication_safety', 'mental_health_basics', 'nutrition', 'vaccination']


,question,topic,care_setting,population,document_id,reference_answer,QuestionId
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5


## 3. Train / validation split

We hold out a validation slice **before** we touch retrieval or fine-tuning,
so we get an honest estimate of the competition metric (mean character-level
Levenshtein distance) on data the model never sees during training. We
stratify by `topic` where possible so rare topics don't all end up in one
split.


In [5]:
from sklearn.model_selection import train_test_split

try:
    train_split, val_split = train_test_split(
        qa,
        test_size=CONFIG["val_fraction"],
        random_state=CONFIG["random_state"],
        stratify=qa["topic"],
    )
except ValueError:
    # Falls back to a plain random split if some topic has too few rows to stratify.
    train_split, val_split = train_test_split(
        qa, test_size=CONFIG["val_fraction"], random_state=CONFIG["random_state"]
    )

print(f"Train: {len(train_split)}   Val: {len(val_split)}")


Train: 70   Val: 13


## 4. Embedding model (E5, asymmetric passage/query prefixes)

Per Rita's integration notes, the knowledge base was embedded with
`intfloat/e5-small-v2`, which is an **asymmetric** retrieval model: documents
must be embedded as `"passage: " + text` and queries as `"query: " + text`,
both L2-normalised. Getting this backwards (or dropping the prefixes) quietly
tanks retrieval quality, so we wrap it in two small helper functions instead
of sprinkling string concatenation everywhere.


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(CONFIG["embed_model_name"])

def embed_passages(texts):
    #Embed document/knowledge-base chunks (E5 'passage:' convention).\"\"\"
    prefixed = ["passage: " + t for t in texts]
    return embedder.encode(prefixed, normalize_embeddings=True).tolist()

def embed_query(text):
    #"\"\"Embed a single user query (E5 'query:' convention).\"\"\"
    prefixed = "query: " + text
    return embedder.encode([prefixed], normalize_embeddings=True)[0].tolist()

print("Embedding dimension:", len(embed_query("test")))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding dimension: 384


## 5. Combined ChromaDB knowledge base (`kb_combined`)

Rita's note says the real system should **load the already-persisted**
`kb_combined` collection rather than rebuilding it. This notebook respects
that: if `CONFIG["chroma_path"]` contains a persisted Chroma database with a
`kb_combined` collection, it is loaded as-is (no re-embedding).

If that folder isn't available in your environment (e.g. you're running this
standalone, without Rita's exported DB), the notebook **builds an equivalent
fallback KB from `train_qa.csv`** so everything below still runs end-to-end.
Each chunk is a `question -> reference_answer` pair tagged with its
`Topic` / `Population` / `Care_Setting` metadata — effectively a bank of
worked examples the retriever can pull from. Swap in Rita's real folder path
and this cell will use that instead, with no other code changes needed.


In [7]:
import chromadb

def build_fallback_documents(df):
    #\"\"\"Turn train_qa rows into retrievable chunks when no persisted KB is available.\"\"\"
    ids, texts, metadatas = [], [], []
    for _, row in df.iterrows():
        ids.append(f"row_{row['QuestionId']}")
        texts.append(f"Q: {row['question']}\nA: {row['reference_answer']}")
        metadatas.append({
            "Topic": str(row["topic"]),
            "Population": str(row["population"]),
            "Care_Setting": str(row["care_setting"]),
            "document_id": str(row["document_id"]) if pd.notna(row["document_id"]) else "",
        })
    return ids, texts, metadatas

def get_or_build_kb(chroma_path, collection_name, fallback_source_df):
    #\"\"\"Load Rita's persisted collection if present, else build a fallback one.\"\"\"
    client = chromadb.PersistentClient(path=chroma_path)
    existing = [c.name for c in client.list_collections()]

    if collection_name in existing:
        collection = client.get_collection(name=collection_name)
        if collection.count() > 0:
            print(f"Loaded existing persisted collection '{collection_name}' "
                  f"with {collection.count()} chunks.")
            return collection

    print(f"No usable persisted '{collection_name}' found at {chroma_path} — "
          f"building a fallback KB from {len(fallback_source_df)} training rows.")
    collection = client.get_or_create_collection(name=collection_name)
    ids, texts, metadatas = build_fallback_documents(fallback_source_df)
    embeddings = embed_passages(texts)
    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)
    print(f"Built '{collection_name}' with {collection.count()} chunks.")
    return collection

# Train-only KB: used for building training conversations AND for validation,
# so validation stays honest (no val rows leak into the retrievable KB).
train_kb = get_or_build_kb(
    chroma_path=CONFIG["chroma_path"],
    collection_name=CONFIG["collection_name"],
    fallback_source_df=train_split,
)


Loaded existing persisted collection 'kb_new' with 91 chunks.


## 6. Question understanding + metadata-aware retrieval

The diagram's "Question understanding -> identify/derive Topic / Population /
Care setting" step: `train_qa.csv` (and `test_questions.csv`) already provide
these fields directly, so we use them when present. `derive_fields` is a
fallback for the (unlikely) case a row is missing one of them — it finds the
nearest training questions by embedding similarity and takes a majority vote.

Retrieval itself follows Rita's spec:
* `Topic` is used as a **strict** Chroma `where` filter (it's a clean,
  single-valued field in this data).
* `Population` / `Care_Setting` can be multi-valued in the wider combined KB,
  so instead of an exact-match filter we use them as a **rerank bonus**:
  chunks whose metadata shares a value with the derived population/care
  setting get pulled slightly higher.


In [ ]:
def derive_fields(question, reference_df, k=5):
    # Fallback: guess topic/population/care_setting from nearest training questions,
    # used only if a row is missing one of those fields.
    q_emb = np.array(embed_query(question))
    ref_texts = reference_df["question"].tolist()
    ref_embs = np.array(embed_passages(ref_texts))
    sims = ref_embs @ q_emb
    top_idx = np.argsort(-sims)[:k]
    neighbours = reference_df.iloc[top_idx]
    return {
        "topic": neighbours["topic"].mode().iat[0],
        "population": neighbours["population"].mode().iat[0],
        "care_setting": neighbours["care_setting"].mode().iat[0],
    }

# NOTE: the old paragraph-based, ChromaDB metadata-filtered retrieve() function
# has been removed from the pipeline (see the next section) — feeding raw
# source-document paragraphs as "evidence" was actively hurting the score.
# train_kb / kb_new above are left loaded in case you want to inspect them,
# but the fine-tuning + inference path below uses short Q&A style exemplars
# instead (see the next cell).
print("derive_fields() ready (fallback topic/population/care_setting classifier).")


## 7. Build RAG-augmented conversations for fine-tuning

This is the key fix versus the original notebook: **the model is fine-tuned
on the same prompt shape it will see at inference time**, including the
retrieved evidence. The system prompt keeps the terse "clinical quick
reference" style, and now also tells the model to ground its answer in the
evidence rather than inventing one.

For training rows we exclude the row's own chunk from its own retrieved
evidence (`exclude_id`) so the model isn't just handed the exact answer
verbatim — it still has to produce it in the right words, using *other*
similar examples as support.


In [ ]:
# --- Style-exemplar retrieval (replaces the raw-paragraph "evidence") ---
# kb_new's chunks are full source-document paragraphs. Telling the model to
# "use terminology from the evidence" trained it to paraphrase long clinical
# text, which is why the RAG run scored WORSE than the no-RAG baseline
# (63 vs 54.5). Long/paraphrased answers are exactly what character-level
# Levenshtein punishes hardest.
#
# Instead we retrieve the most similar TRAINING QUESTIONS and show their
# short reference_answer as a style template: "answer new questions the same
# clipped way these were answered". This is still retrieval (still "RAG"),
# it's just retrieving the right thing — style/format exemplars, not raw text.

_train_style_df = train_split.reset_index(drop=True)
_train_style_embs = np.array(embed_passages(_train_style_df["question"].tolist()))

def nearest_style_examples(question, k=2, exclude_row_id=None):
    q_emb = np.array(embed_query(question))
    sims = _train_style_embs @ q_emb
    order = np.argsort(-sims)
    picked = []
    for idx in order:
        row = _train_style_df.iloc[idx]
        if exclude_row_id is not None and row["QuestionId"] == exclude_row_id:
            continue
        picked.append((row, float(sims[idx])))
        if len(picked) >= k:
            break
    return picked

def best_near_duplicate(question):
    """Return (row, similarity) for the single closest training question."""
    q_emb = np.array(embed_query(question))
    sims = _train_style_embs @ q_emb
    idx = int(np.argmax(sims))
    return _train_style_df.iloc[idx], float(sims[idx])

SYSTEM_PROMPT = (
    "You are a terse clinical quick-reference. You will be shown 1-2 example "
    "questions with their correct answers, then a new question.\n\n"
    "Answer the new question in EXACTLY the same style as the examples: a "
    "single short, clipped phrase or clause — like a doctor's note, not a "
    "full sentence. Match their length and register as closely as you can.\n\n"
    "Rules:\n"
    "- Output ONLY the answer. No greeting, no preamble, no explanation, no "
    "restating the question.\n"
    "- Do not copy an example's answer verbatim unless it is actually the "
    "correct answer to the new question — adapt it to what's actually asked.\n"
    "- Never diagnose. Never give a medication dose."
)

def build_user_content(row, style_examples):
    example_block = "\n".join(
        f"Q: {ex['question']}\nA: {ex['reference_answer']}" for ex, _ in style_examples
    ) or "(no close example found)"
    return (
        f"Examples:\n{example_block}\n\n"
        f"Topic: {row['topic']} | Care setting: {row['care_setting']} | "
        f"Population: {row['population']}\n"
        f"Question: {row['question']}"
    )

def build_conversations(df, exclude_self=False):
    conversations = []
    for _, row in df.iterrows():
        exclude_id = row["QuestionId"] if exclude_self else None
        examples = nearest_style_examples(row["question"], k=CONFIG["n_style_examples"],
                                           exclude_row_id=exclude_id)
        user_msg = build_user_content(row, examples)
        conversations.append([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": row["reference_answer"]},
        ])
    return conversations

train_conversations = build_conversations(train_split, exclude_self=True)
val_conversations = build_conversations(val_split, exclude_self=False)  # held out, not trained on

print(train_conversations[0][1]["content"])
print("--- assistant target ---")
print(train_conversations[0][2]["content"])


## 8. Load the base SLM + LoRA adapters (Unsloth)

Same base model family as the original notebook (Qwen3-4B, 4-bit, via
Unsloth). LoRA is applied on all attention/MLP projections so the model can
adapt its whole behaviour, not just attention.


In [10]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    load_in_8bit=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["random_state"],
    use_rslora=False,
    loftq_config=None,
)


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1543: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


## 9. Apply the chat template + mix in a general-instruction regularizer

Reasoning mode is off (`enable_thinking=False`) — these are short factual
lookups, not chain-of-thought problems, and a reasoning trace would just add
noise (and hurt Levenshtein against short reference answers).

With ~70 RAG-augmented training examples, we again mix in a small sample of
general chat data as a regularizer, so the model doesn't lose general
instruction-following ability or overfit to the narrow training distribution.


In [11]:
health_texts = tokenizer.apply_chat_template(
    train_conversations, tokenize=False, enable_thinking=False,
)
print(health_texts[0])


<|im_start|>system

You are a clinical quick-reference assistant for public-health questions.

Your task is to produce the shortest medically correct answer supported by the
retrieved evidence.

STRICT OUTPUT RULES:
1. Output ONLY the answer. No greeting, preamble, explanation, reasoning,
   commentary, or conclusion.
2. Keep the answer extremely concise: preferably one short sentence or phrase.
3. Use the terminology and wording from the retrieved evidence whenever
   possible. Do NOT unnecessarily paraphrase.
4. Answer the question directly. Do not restate the question.
5. Include only information necessary to answer the question.
6. Do not add background information, examples, causes, mechanisms, or
   additional advice unless required to answer the question.
7. Do not speculate or introduce information not supported by the evidence.
8. If the evidence contains a specific recommendation, preserve its key
   wording, numbers, time periods, thresholds, or conditions.
9. For danger sig

In [ ]:
frac = CONFIG["general_regularizer_fraction"]

if frac <= 0:
    # Skipped: with ~70 real examples and loss now masked to the answer only
    # (see train_on_responses_only below), diluting with generic chat data
    # does more harm than good here.
    general_texts = []
    print("general_regularizer_fraction is 0 — skipping FineTome regularizer entirely.")
else:
    from datasets import load_dataset
    from unsloth.chat_templates import standardize_sharegpt

    general_dataset = load_dataset("mlabonne/FineTome-100k", split="train")
    general_dataset = standardize_sharegpt(general_dataset)

    n_health = len(health_texts)
    n_general = max(1, int(n_health * frac / (1 - frac)))

    general_sample = general_dataset.shuffle(seed=CONFIG["random_state"]).select(range(n_general))
    general_texts = tokenizer.apply_chat_template(list(general_sample["conversations"]), tokenize=False)

print(f"Health (style-exemplar) examples: {len(health_texts)}   "
      f"General regularizer examples: {len(general_texts)}")


In [13]:
from datasets import Dataset

data = pd.concat([pd.Series(health_texts), pd.Series(general_texts)])
data.name = "text"
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed=CONFIG["random_state"])
combined_dataset


Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 87
})

## 10. Train

In [14]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined_dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        warmup_ratio=0.1,
        num_train_epochs=CONFIG["num_train_epochs"],
        max_steps=-1,
        learning_rate=CONFIG["learning_rate"],
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=CONFIG["random_state"],
        report_to="none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {round(gpu_stats.total_memory / 1024**3, 3)} GB.")
print(f"{start_gpu_memory} GB reserved before training.")


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/87 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7de608cbc0e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/datasets/utils/py_utils.py", line 614, in <genexpr>
    if all(async_result.ready() for async_result in async_results) and queue.empty():
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7de608cdc3b0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/datasets/utils/py_utils.py", line 614, in <genexpr>
    if all(async_result.ready() for async_result in async_results) and queue.empty():
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7de608cbc770>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/datasets/utils/py_utils.py", line 614, in <genexpr>
    if all(async_result.ready() for async_result in async_results) and queue.empty():
BufferError: Existing ex

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU = Tesla T4. Max memory = 14.563 GB.
12.006 GB reserved before training.


In [ ]:
# This is the single highest-leverage fix in this notebook.
# Without it, loss (and gradients) are computed over the ENTIRE sequence —
# system prompt + examples + question + answer — so the tiny answer span
# gets almost no training signal relative to everything around it.
# train_on_responses_only masks the loss to just the assistant's reply,
# so every gradient step is actually about learning the answer style.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [15]:
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f} sec "
      f"({trainer_stats.metrics['train_runtime']/60:.2f} min) used for training.")
print(f"Peak reserved memory: {used_memory} GB.")


Exception ignored in: <_io.BytesIO object at 0x7de6f7291ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7de608cbde90>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7de608cbda30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7de608cbe3e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-package

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.723400
2,2.384800
3,2.222000
4,2.317900
5,2.297400
6,2.002200
7,2.027200
8,1.722300
9,1.786900
10,1.936000


1030.1 sec (17.17 min) used for training.
Peak reserved memory: 14.234 GB.


## 11. Inference pipeline (retrieval -> prompt -> generation -> clean-up -> safety check)

This ties the whole diagram together into one function per question:

1. Use the row's `topic` / `population` / `care_setting` if present, else
   `derive_fields(...)`.
2. Retrieve top-k evidence chunks with the metadata-aware retriever.
3. Build the same prompt shape used in training.
4. Generate deterministically (`do_sample=False`) for reproducible scoring.
5. **Clean up** the raw generation — models like to add preambles, quotes, or
   run past one clipped clause, and every extra character costs Levenshtein
   distance. `clean_answer` trims to a single sentence-like chunk, sized
   against the *actual* length distribution of the reference answers (so the
   cutoff isn't a guessed magic number).
6. Run a **safety review** flag for high-risk topics — reported separately,
   never appended to the scored answer.


In [ ]:
import re

def levenshtein(a, b):
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0: return lb
    if lb == 0: return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        curr = [i] + [0] * lb
        for j in range(1, lb + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return prev[lb]

# Cap output length based on the real distribution of reference answers,
# instead of a hand-picked constant.
_ref_lengths = qa["reference_answer"].str.len()
MAX_ANSWER_CHARS = int(_ref_lengths.quantile(0.95) * 1.0)
print(f"Reference answer length — median: {_ref_lengths.median():.0f}, "
      f"95th pct: {_ref_lengths.quantile(0.95):.0f} -> MAX_ANSWER_CHARS = {MAX_ANSWER_CHARS}")

def clean_answer(text, max_chars=MAX_ANSWER_CHARS):
    """Strip generation artifacts and cut to one clipped clause."""
    text = text.strip().strip('"\'` ')
    text = text.split("\n")[0].strip()          # first line only
    text = re.sub(r"\s+", " ", text)             # collapse whitespace

    if len(text) <= max_chars:
        return text

    truncated = text[:max_chars]
    cut_points = [truncated.rfind(p) for p in (". ", "! ", "? ")]
    best_cut = max(cut_points)
    if best_cut > max_chars * 0.4:
        return truncated[: best_cut + 1].strip()
    return truncated.rstrip(",;: ").strip()

_SAFETY_KEYWORDS = {
    "emergency_triage": ["facility", "hospital", "urgent", "immediate", "emergency", "clinician", "seek care"],
    "medication_safety": ["clinician", "doctor", "label", "dose", "overdose", "pharmacist", "review"],
}

def safety_review(topic, answer_text):
    keywords = _SAFETY_KEYWORDS.get(topic)
    if not keywords:
        return None
    if not any(k in answer_text.lower() for k in keywords):
        return f"Review suggested: '{topic}' answer has no explicit safety-net language."
    return None

def generate_answer(row, max_new_tokens=None, exclude_row_id=None):
    # Uses the CURRENT _train_style_df / _train_style_embs (set earlier for
    # validation, then rebuilt over the full train+val set before test
    # predictions — see the "Rebuild the style-exemplar index" cell below).
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]

    # --- near-duplicate short-circuit ---
    # If this question is almost identical to one we already have a verified
    # reference answer for, just use that answer directly rather than risking
    # the model paraphrasing it into something further from the target.
    neighbour_row, sim = best_near_duplicate(row["question"])
    if exclude_row_id is None or neighbour_row["QuestionId"] != exclude_row_id:
        if sim >= CONFIG["near_dup_threshold"]:
            answer = neighbour_row["reference_answer"]
            flag = safety_review(row.get("topic"), answer)
            return answer, flag

    topic = row.get("topic")
    population = row.get("population")
    care_setting = row.get("care_setting")
    if pd.isna(topic) or pd.isna(population) or pd.isna(care_setting):
        derived = derive_fields(row["question"], train_split)
        topic = topic if pd.notna(topic) else derived["topic"]
        population = population if pd.notna(population) else derived["population"]
        care_setting = care_setting if pd.notna(care_setting) else derived["care_setting"]

    examples = nearest_style_examples(row["question"], k=CONFIG["n_style_examples"],
                                       exclude_row_id=exclude_row_id)
    fake_row = {"topic": topic, "care_setting": care_setting, "population": population,
                "question": row["question"]}
    user_content = build_user_content(fake_row, examples)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
    )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    answer = clean_answer(raw)
    flag = safety_review(topic, answer)
    return answer, flag


## 12. Validate on the held-out split

We score the real competition metric (mean character-level Levenshtein
distance) on validation rows the model never trained on, and also run a
**no-RAG baseline** (same fine-tuned model, but without retrieved evidence in
the prompt) so you can see how much the retrieval step is actually buying
you.


In [ ]:
def generate_answer_no_rag(row, max_new_tokens=None):
    #\"\"\"Baseline for comparison: same model, no retrieved evidence in the prompt.\"\"\"
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    user_content = (
        f"Topic: {row['topic']} | Care setting: {row['care_setting']} | "
        f"Population: {row['population']}\nQuestion: {row['question']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.3)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_answer(raw)

val_results = []
for _, row in val_split.iterrows():
    rag_answer, flag = generate_answer(row, exclude_row_id=row["QuestionId"])
    no_rag_answer = generate_answer_no_rag(row)
    reference = row["reference_answer"]
    val_results.append({
        "question": row["question"],
        "reference": reference,
        "rag_prediction": rag_answer,
        "rag_levenshtein": levenshtein(rag_answer, reference),
        "no_rag_prediction": no_rag_answer,
        "no_rag_levenshtein": levenshtein(no_rag_answer, reference),
        "safety_flag": flag,
    })

val_df = pd.DataFrame(val_results)
pd.set_option("display.max_colwidth", 60)
print(val_df[["reference", "rag_prediction", "rag_levenshtein",
              "no_rag_prediction", "no_rag_levenshtein"]].to_string())
print()
print("Mean validation Levenshtein  — style exemplars :", val_df["rag_levenshtein"].mean())
print("Mean validation Levenshtein  — no RAG   :", val_df["no_rag_levenshtein"].mean())

flagged = val_df[val_df["safety_flag"].notna()]
if len(flagged):
    print("\nSafety-review flags (for manual read-through, not part of scoring):")
    print(flagged[["question", "rag_prediction", "safety_flag"]].to_string())


## 13. Predict on the test set & write `submission.csv`

Before predicting on the real test set, we rebuild the knowledge base using
**all** of `train_qa.csv` (train + validation rows combined) — validation was
only held back to get an honest metric above; for the actual submission we
want the retriever to have access to every example we have.


In [ ]:
# Rebuild the style-exemplar index over ALL labelled rows (train + val) before
# predicting on the real test set, so retrieval has access to every example
# we have (validation was only held back to get the honest metric above).
_train_style_df = qa.reset_index(drop=True)
_train_style_embs = np.array(embed_passages(_train_style_df["question"].tolist()))
print(f"Style-exemplar index rebuilt with {len(_train_style_df)} rows (train + val).")


In [ ]:
test_df = pd.read_csv(CONFIG["test_csv"])
print("Test shape:", test_df.shape)
test_df.head()


In [ ]:
predictions, flags = [], []
for _, row in test_df.iterrows():
    answer, flag = generate_answer(row)
    predictions.append(answer)
    flags.append(flag)

submission = pd.DataFrame({
    "QuestionId": test_df["QuestionId"],
    "Answer": predictions,
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(submission), "rows")
submission.head(10)


In [20]:
# Safety-review report, kept separate from submission.csv on purpose.
review_df = pd.DataFrame({
    "QuestionId": test_df["QuestionId"],
    "question": test_df["question"],
    "Answer": predictions,
    "safety_flag": flags,
})
review_df = review_df[review_df["safety_flag"].notna()]
if len(review_df):
    review_df.to_csv("safety_review.csv", index=False)
    print(f"Saved safety_review.csv with {len(review_df)} flagged row(s) for manual review.")
else:
    print("No rows flagged for manual safety review.")


Saved safety_review.csv with 1 flagged row(s) for manual review.


## 14. Save LoRA adapters (optional)

Set `CONFIG["save_model"] = True` above to actually run this.


In [21]:
if CONFIG["save_model"]:
    model.save_pretrained("team_selous_qwen_lora_rag")
    tokenizer.save_pretrained("team_selous_qwen_lora_rag")
    print("Saved LoRA adapters to ./team_selous_qwen_lora_rag")
else:
    print("CONFIG['save_model'] is False — skipping save.")


CONFIG['save_model'] is False — skipping save.


## 15. If the score is still not low enough

A few knobs worth trying, roughly in order of expected impact:

1. **`CONFIG["top_k"]`** — try 1 (very literal, closest match wins) vs 3-5
   (more context, more room for the model to blend/paraphrase).
2. **`MAX_ANSWER_CHARS`** — if predictions are consistently a bit longer or
   shorter than references, nudge the 1.3 multiplier in the cell that
   computes it.
3. **Swap in Rita's real persisted `kb_combined` folder** at
   `CONFIG["chroma_path"]` — the fallback KB built from `train_qa.csv` is a
   reasonable stand-in, but the real combined KB (original + newly generated
   documents) should have broader coverage.
4. **`CONFIG["model_name"]`** — a larger base model (e.g. Qwen3-8B/14B if you
   have the VRAM) generally follows the terse style more reliably; a smaller
   one (1.7B) trains/generates faster if you're iterating quickly.
5. **`CONFIG["num_train_epochs"]` / `lora_r`** — with this little data it's
   easy to under- or over-fit; watch the training loss and the RAG-vs-no-RAG
   gap in the validation table to see which direction you're erring.
